In [1]:
import os
import shutil
import zipfile
from pathlib import Path
import pandas as pd
from tqdm import tqdm

In [2]:

root_dir = r'Y:\ZHL\isds\PS\task0722'
merge_dir = os.path.join(root_dir, 'merge_dir')
root_folder_id = '1ygoXiZGNBcykjbrEu360HnwZTqiRH0xP'
client_secret = r"E:\data\202502_signboard\data_annotation\docs\client_secret.json"
token_path = 'token.json'
SCOPES = ['https://www.googleapis.com/auth/drive.readonly']

In [6]:
import os
import io
from concurrent.futures import ThreadPoolExecutor
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from google.auth.transport.requests import Request


def authenticate_with_google(token_path, client_secret_path):
    creds = None

    if os.path.exists(token_path):
        creds = Credentials.from_authorized_user_file(token_path, SCOPES)

    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file(client_secret_path, SCOPES)
            creds = flow.run_local_server(port=0)
        with open(token_path, 'w') as token_file:
            token_file.write(creds.to_json())

    service = build('drive', 'v3', credentials=creds)
    return service


def download_large_file(service, file_id, file_path):
    os.makedirs(os.path.dirname(file_path), exist_ok=True)
    if os.path.exists(file_path):
        print(f"⚠️ 已存在，跳过: {file_path}")
        return
    print(f"⬇️ Downloading {file_path}")
    request = service.files().get_media(fileId=file_id)
    with io.FileIO(file_path, 'wb') as fh:
        downloader = MediaIoBaseDownload(fh, request)
        done = False
        while not done:
            status, done = downloader.next_chunk()
            if status:
                print(f"⬇️ Downloading {file_path}: {int(status.progress() * 100)}%")
    print(f"✅ Finished: {file_path}")

def download_folder_recursive(service, folder_id, save_path):
    os.makedirs(save_path, exist_ok=True)
    query = f"'{folder_id}' in parents and trashed = false"
    results = service.files().list(q=query, fields="files(id, name, mimeType)").execute()
    items = results.get('files', [])

    for item in items:
        file_id = item['id']
        file_name = item['name']
        file_mime = item['mimeType']
        full_path = os.path.join(save_path, file_name)

        if file_mime == 'application/vnd.google-apps.folder':
            download_folder_recursive(service, file_id, full_path)
        else:
            download_large_file(service, file_id, full_path)

def download_subfolder_task(folder_obj, root_save_path, token_path, client_secret_path):
    # 每个线程都单独认证，避免多线程共享service导致问题
    service = authenticate_with_google(token_path, client_secret_path)
    folder_id = folder_obj['id']
    folder_name = folder_obj['name']
    target_path = os.path.join(root_save_path, folder_name)
    print(f"\n📁 Starting folder: {folder_name}")
    download_folder_recursive(service, folder_id, target_path)

def download_all_subfolders_parallel(token_path, client_secret_path, root_folder_id, save_dir):
    os.makedirs(save_dir, exist_ok=True)

    # 主线程先获取子文件夹列表
    service = authenticate_with_google(token_path, client_secret_path)
    query = f"'{root_folder_id}' in parents and trashed = false and mimeType = 'application/vnd.google-apps.folder'"
    results = service.files().list(q=query, fields="files(id, name)").execute()
    folders = results.get('files', [])

    print(f"将并发下载 {len(folders)} 个子文件夹...\n")

    with ThreadPoolExecutor(max_workers=len(folders)) as executor:
        for folder in folders:
            executor.submit(download_subfolder_task, folder, save_dir, token_path, client_secret_path)



In [7]:

download_all_subfolders_parallel(token_path, client_secret, root_folder_id, root_dir)

将并发下载 6 个子文件夹...


📁 Starting folder: 13-14-35

📁 Starting folder: 15-41-23

📁 Starting folder: 15-13-01

📁 Starting folder: 11-40-29

📁 Starting folder: 11-03-12

📁 Starting folder: 12-26-29
⬇️ Downloading Y:\ZHL\isds\PS\task0722\11-03-12\rectified_images.zip
⬇️ Downloading Y:\ZHL\isds\PS\task0722\11-40-29\rectified_images.zip
⬇️ Downloading Y:\ZHL\isds\PS\task0722\15-13-01\rectified_images.zip
⬇️ Downloading Y:\ZHL\isds\PS\task0722\12-26-29\rectified_images.zip
⬇️ Downloading Y:\ZHL\isds\PS\task0722\15-41-23\rectified_images.zip
⬇️ Downloading Y:\ZHL\isds\PS\task0722\13-14-35\rectified_images.zip
⬇️ Downloading Y:\ZHL\isds\PS\task0722\13-14-35\rectified_images.zip: 0%
⬇️ Downloading Y:\ZHL\isds\PS\task0722\15-41-23\rectified_images.zip: 1%
⬇️ Downloading Y:\ZHL\isds\PS\task0722\11-40-29\rectified_images.zip: 1%
⬇️ Downloading Y:\ZHL\isds\PS\task0722\15-13-01\rectified_images.zip: 2%
⬇️ Downloading Y:\ZHL\isds\PS\task0722\12-26-29\rectified_images.zip: 1%
⬇️ Downloading Y:\ZHL\isds\PS

In [8]:
def uzip_dirs(root_dir):
    sub_dir_list = os.listdir(root_dir)
    for sub_dir_name in sub_dir_list:
        sub_dir = os.path.join(root_dir, sub_dir_name)
        zip_path = os.path.join(sub_dir, 'rectified_images.zip')
        if not os.path.exists(zip_path):
            print(f'{zip_path} not exists')
        else:
            print(f'{zip_path} unzip...')
            with zipfile.ZipFile(zip_path, 'r') as zip_ref:
                zip_ref.extractall(zip_path.replace('.zip', ''))
            print(f'{zip_path} done\n')

In [9]:
uzip_dirs(root_dir)

Y:\ZHL\isds\PS\task0722\11-03-12\rectified_images.zip unzip...
Y:\ZHL\isds\PS\task0722\11-03-12\rectified_images.zip done

Y:\ZHL\isds\PS\task0722\11-40-29\rectified_images.zip unzip...
Y:\ZHL\isds\PS\task0722\11-40-29\rectified_images.zip done

Y:\ZHL\isds\PS\task0722\12-26-29\rectified_images.zip unzip...
Y:\ZHL\isds\PS\task0722\12-26-29\rectified_images.zip done

Y:\ZHL\isds\PS\task0722\13-14-35\rectified_images.zip unzip...
Y:\ZHL\isds\PS\task0722\13-14-35\rectified_images.zip done

Y:\ZHL\isds\PS\task0722\15-13-01\rectified_images.zip unzip...
Y:\ZHL\isds\PS\task0722\15-13-01\rectified_images.zip done

Y:\ZHL\isds\PS\task0722\15-41-23\rectified_images.zip unzip...
Y:\ZHL\isds\PS\task0722\15-41-23\rectified_images.zip done



In [10]:
from img_preprocess import select_img
from deduplication_demo import filter_deduplication

def process_dirs(root_dir):
    sub_dirs = os.listdir(root_dir)
    for idx, sub_name in enumerate(sub_dirs):
        sub_dir = os.path.join(root_dir, sub_name)
        if not os.path.isdir(sub_dir) or not sub_name.startswith('1'):
            continue
        cam_name_list = ['cam_DA4930148', 'cam_DA5148680', 'cam_DA5148683', 'cam_DA5324645', 'cam_DA5324655', 'cam_DA6102933']
        for cam_name in cam_name_list:
            image_dir_src = os.path.join(sub_dir, 'rectified_images', 'rectified_images', cam_name)
            if not os.path.exists(image_dir_src):
                print(f'{image_dir_src} not exists')
            else:
                print(f'{image_dir_src} selecting...')
                image_dir_select = image_dir_src+'_select'
                shutil.rmtree(image_dir_select) if os.path.exists(image_dir_select) else None
                select_img(image_dir_src, image_dir_select, gap=30)
                print(f'{image_dir_select} filtering...')
                image_dir_filter = image_dir_src+'_filter' 
                shutil.rmtree(image_dir_filter) if os.path.exists(image_dir_filter) else None
                filter_deduplication(image_dir_select, image_dir_filter)
                print(f'{image_dir_filter} done\n')

In [11]:
process_dirs(root_dir)

Y:\ZHL\isds\PS\task0722\11-03-12\rectified_images\rectified_images\cam_DA4930148 selecting...


100%|██████████| 110/110 [00:04<00:00, 24.11it/s]


Y:\ZHL\isds\PS\task0722\11-03-12\rectified_images\rectified_images\cam_DA4930148_select filtering...


100%|██████████| 110/110 [00:10<00:00, 10.98it/s]


Copied: cam_image_20250722110323600.jpg (Max similarity: 0.28)
Copied: cam_image_20250722110326600.jpg (Max similarity: 0.24)
Copied: cam_image_20250722110329599.jpg (Max similarity: 0.29)
Copied: cam_image_20250722110332600.jpg (Max similarity: 0.30)
Copied: cam_image_20250722110335600.jpg (Max similarity: 0.30)
Copied: cam_image_20250722110338599.jpg (Max similarity: 0.27)
Copied: cam_image_20250722110341600.jpg (Max similarity: 0.29)
Copied: cam_image_20250722110344599.jpg (Max similarity: 0.30)
Copied: cam_image_20250722110347600.jpg (Max similarity: 0.30)
Copied: cam_image_20250722110350600.jpg (Max similarity: 0.29)
Copied: cam_image_20250722110353600.jpg (Max similarity: 0.30)
Copied: cam_image_20250722110356600.jpg (Max similarity: 0.31)
Copied: cam_image_20250722110359600.jpg (Max similarity: 0.30)
Copied: cam_image_20250722110402600.jpg (Max similarity: 0.29)
Copied: cam_image_20250722110405600.jpg (Max similarity: 0.32)
Copied: cam_image_20250722110408599.jpg (Max similarity

...

Copied: cam_image_20250722154131299.jpg (Max similarity: 0.36)
Copied: cam_image_20250722154134299.jpg (Max similarity: 0.36)
Copied: cam_image_20250722154137299.jpg (Max similarity: 0.31)
Copied: cam_image_20250722154140299.jpg (Max similarity: 0.31)
Copied: cam_image_20250722154143300.jpg (Max similarity: 0.35)
Copied: cam_image_20250722154146300.jpg (Max similarity: 0.33)
Copied: cam_image_20250722154149300.jpg (Max similarity: 0.37)
Copied: cam_image_20250722154152300.jpg (Max similarity: 0.40)
Copied: cam_image_20250722154155299.jpg (Max similarity: 0.40)
Copied: cam_image_20250722154158299.jpg (Max similarity: 0.46)
Copied: cam_image_20250722154252300.jpg (Max similarity: 0.36)
Copied: cam_image_20250722154255299.jpg (Max similarity: 0.31)
Copied: cam_image_20250722154258299.jpg (Max similarity: 0.39)
Copied: cam_image_20250722154301299.jpg (Max similarity: 0.40)
Copied: cam_image_20250722154304300.jpg (Max similarity: 0.40)
Copied: cam_image_20250722154307300.jpg (Max similarity

100%|██████████| 289/289 [00:06<00:00, 44.09it/s]


Y:\ZHL\isds\PS\task0722\15-41-23\rectified_images\rectified_images\cam_DA6102933_select filtering...


100%|██████████| 289/289 [00:20<00:00, 13.94it/s]


Copied: cam_image_20250722154137299.jpg (Max similarity: 0.43)
Copied: cam_image_20250722154140299.jpg (Max similarity: 0.39)
Copied: cam_image_20250722154143300.jpg (Max similarity: 0.26)
Copied: cam_image_20250722154146300.jpg (Max similarity: 0.27)
Copied: cam_image_20250722154149300.jpg (Max similarity: 0.26)
Copied: cam_image_20250722154152300.jpg (Max similarity: 0.31)
Copied: cam_image_20250722154155299.jpg (Max similarity: 0.29)
Copied: cam_image_20250722154158299.jpg (Max similarity: 0.46)
Copied: cam_image_20250722154252300.jpg (Max similarity: 0.33)
Copied: cam_image_20250722154255299.jpg (Max similarity: 0.18)
Copied: cam_image_20250722154258299.jpg (Max similarity: 0.26)
Copied: cam_image_20250722154301299.jpg (Max similarity: 0.29)
Copied: cam_image_20250722154304300.jpg (Max similarity: 0.26)
Copied: cam_image_20250722154307300.jpg (Max similarity: 0.30)
Copied: cam_image_20250722154310300.jpg (Max similarity: 0.30)
Copied: cam_image_20250722154313300.jpg (Max similarity

In [12]:
def img_merge(input_dir, output_dir):
    sub_dirs = os.listdir(input_dir)
    if 'merge_dir' in sub_dirs:
        sub_dirs.remove('merge_dir')
    os.makedirs(output_dir, exist_ok=True)
    for sub_name in sub_dirs:
        sub_dir = os.path.join(input_dir, sub_name)
        if not os.path.isdir(sub_dir):
            continue
        cam_name_list = ['cam_DA4930148', 'cam_DA5148680', 'cam_DA5148683', 'cam_DA5324645', 'cam_DA5324655', 'cam_DA6102933']
        for cam_name in cam_name_list:
            image_dir_src = os.path.join(sub_dir, 'rectified_images', 'rectified_images', cam_name+'_filter')
            if not os.path.exists(image_dir_src):
                print(f'{image_dir_src} not exists')
            else:
                img_list = os.listdir(image_dir_src)
                for img_name in tqdm(img_list):
                    img_path_src = os.path.join(image_dir_src, img_name)
                    img_path_dst = os.path.join(output_dir, cam_name+'_'+img_name)
                    shutil.copyfile(img_path_src, img_path_dst)
    


In [13]:
img_merge(root_dir, merge_dir)

100%|██████████| 159/159 [00:02<00:00, 53.01it/s]


In [14]:
print(len(os.listdir(merge_dir)))

3771


In [15]:
import zipfile
import os

def zip_folder_to_path(source_folder, destination_zip):
    with zipfile.ZipFile(destination_zip, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, dirs, files in os.walk(source_folder):
            for file in files:
                file_path = os.path.join(root, file)
                # 在zip文件中创建相对路径
                arcname = os.path.relpath(file_path, start=source_folder)
                zipf.write(file_path, arcname)
    
    print(f"zip '{source_folder}' to '{destination_zip}'")

zip_folder_to_path(
    source_folder=merge_dir,
    destination_zip=os.path.join(root_dir, os.path.basename(root_dir)+'.zip')
)

zip 'Y:\ZHL\isds\PS\task0722\merge_dir' to 'Y:\ZHL\isds\PS\task0722\task0722.zip'
